In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, threading, platform
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster       import KMeans
from sklearn.ensemble      import IsolationForest
from sklearn.metrics       import silhouette_score

# -- DPI haute resolution (Windows / portable) ---------------------------------
if platform.system() == "Windows":
    try:
        import ctypes
        ctypes.windll.shcore.SetProcessDpiAwareness(2)
    except Exception:
        try:
            ctypes.windll.user32.SetProcessDPIAware()
        except Exception:
            pass

In [ ]:
# Cellule 1 - Algorithmes ML (best_k, analyser_data, visualisations)

In [ ]:
# -- best_k -------------------------------------------------------------------
def best_k(X_scaled, k_range=range(2, 7)):
    scores = {}
    for k in k_range:
        try:
            lbl = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
            scores[k] = silhouette_score(X_scaled, lbl)
        except Exception:
            pass
    return max(scores, key=scores.get) if scores else 3


# -- analyser_data ------------------------------------------------------------
def analyser_data(X_df, k=3, contamination=0.05, auto_k=False):
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_df)
    if auto_k:
        k = best_k(X_scaled)
    pca     = PCA(n_components=2)
    coords  = pca.fit_transform(X_scaled)
    var_exp = pca.explained_variance_ratio_.sum()
    labels  = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    try:
        sil = silhouette_score(X_scaled, labels)
    except Exception:
        sil = 0.0
    anom   = IsolationForest(contamination=contamination, random_state=42).fit_predict(X_scaled)
    n_anom = (anom == -1).sum()
    return {
        "coords": coords, "labels": labels, "anomalies": anom,
        "k": k, "n_anomalies": n_anom,
        "variance_explained": var_exp, "silhouette_score": sil,
        "X_df": X_df,
    }


# -- creer_visualisations (vue ensemble, conserve de v1) ----------------------
def creer_visualisations(resultats, titre="Analyse ML"):
    coords  = resultats["coords"]; labels = resultats["labels"]
    anom    = resultats["anomalies"]; X_df = resultats["X_df"]
    k       = resultats["k"]; pal = plt.cm.tab10.colors
    fig = Figure(figsize=(17, 5))
    ax1 = fig.add_subplot(131)
    for i in range(k):
        m = labels == i
        ax1.scatter(coords[m,0], coords[m,1], color=pal[i%10],
                    label=f"Cluster {i}", s=25, alpha=0.75)
    ax1.set_title(f"KMeans (k={k})", fontweight="bold")
    ax1.set_xlabel("PC1"); ax1.set_ylabel("PC2")
    ax1.legend(fontsize=8); ax1.grid(True, alpha=0.25)
    ax2 = fig.add_subplot(132)
    corr = X_df.corr()
    im   = ax2.imshow(corr, cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
    ax2.set_xticks(range(len(corr.columns))); ax2.set_yticks(range(len(corr.columns)))
    ax2.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
    ax2.set_yticklabels(corr.columns, fontsize=8)
    if len(corr.columns) <= 8:
        for r in range(len(corr.columns)):
            for c in range(len(corr.columns)):
                ax2.text(c, r, f"{corr.iloc[r,c]:.2f}", ha="center", va="center",
                         color="black", fontsize=7)
    ax2.set_title("Correlation", fontweight="bold"); fig.colorbar(im, ax=ax2)
    ax3 = fig.add_subplot(133)
    ax3.scatter(coords[anom==1,0], coords[anom==1,1], label="Normal", s=18, alpha=0.45, c="steelblue")
    ax3.scatter(coords[anom==-1,0], coords[anom==-1,1],
                label=f"Anomalie ({(anom==-1).sum()})", s=55, marker="X",
                color="red", edgecolors="black", linewidths=0.8)
    ax3.set_title("Isolation Forest", fontweight="bold")
    ax3.set_xlabel("PC1"); ax3.set_ylabel("PC2")
    ax3.legend(fontsize=8); ax3.grid(True, alpha=0.25)
    fig.suptitle(titre, fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    return fig


# -- creer_visualisations_v2 (4 figures individuelles pour les onglets) --------
def creer_visualisations_v2(resultats, titre="Analyse ML"):
    coords = resultats["coords"]; labels = resultats["labels"]
    anom   = resultats["anomalies"]; X_df = resultats["X_df"]
    k      = resultats["k"]; pal = plt.cm.tab10.colors

    rc = {
        "figure.facecolor": "white", "axes.facecolor": "#fafbfc",
        "axes.edgecolor": "#d0d7de", "axes.labelcolor": "#24292f",
        "xtick.color": "#57606a", "ytick.color": "#57606a",
        "text.color": "#24292f", "grid.color": "#e8eaed",
        "axes.titlecolor": "#24292f",
        "axes.spines.top": False, "axes.spines.right": False,
    }

    def draw_cluster(ax):
        for i in range(k):
            m = labels == i
            ax.scatter(coords[m,0], coords[m,1], color=pal[i%10],
                       label=f"Cluster {i}  (n={m.sum()})", s=32, alpha=0.78,
                       edgecolors="white", linewidths=0.4)
        ax.set_title(f"KMeans Clustering  (k={k})", fontweight="bold", fontsize=11)
        ax.set_xlabel("Composante 1  (PC1)", fontsize=9)
        ax.set_ylabel("Composante 2  (PC2)", fontsize=9)
        ax.legend(fontsize=8, framealpha=0.8); ax.grid(True, alpha=0.35, linestyle="--")

    def draw_corr(fig_r, ax):
        corr = X_df.corr()
        im   = ax.imshow(corr, cmap="coolwarm", aspect="auto",
                         vmin=-1, vmax=1, interpolation="nearest")
        ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=9)
        ax.set_yticklabels(corr.columns, fontsize=9)
        if len(corr.columns) <= 10:
            for r in range(len(corr.columns)):
                for c in range(len(corr.columns)):
                    v = corr.iloc[r, c]
                    ax.text(c, r, f"{v:.2f}", ha="center", va="center",
                            color="white" if abs(v)>0.6 else "#24292f", fontsize=8)
        ax.set_title("Matrice de Correlation (Pearson)", fontweight="bold", fontsize=11)
        fig_r.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    def draw_anom(ax):
        nn = (anom==1).sum(); na = (anom==-1).sum()
        ax.scatter(coords[anom==1,0], coords[anom==1,1],
                   label=f"Normal  (n={nn})", s=22, alpha=0.45, c="#0969da")
        ax.scatter(coords[anom==-1,0], coords[anom==-1,1],
                   label=f"Anomalie  (n={na})", s=65, marker="X",
                   color="#cf222e", edgecolors="#24292f", linewidths=0.8)
        ax.set_title("Isolation Forest  - Detection d'Anomalies",
                     fontweight="bold", fontsize=11)
        ax.set_xlabel("Composante 1  (PC1)", fontsize=9)
        ax.set_ylabel("Composante 2  (PC2)", fontsize=9)
        ax.legend(fontsize=8, framealpha=0.8); ax.grid(True, alpha=0.35, linestyle="--")

    with plt.rc_context(rc):
        # Vue d'ensemble
        fig_all = Figure(figsize=(17, 5))
        ax1 = fig_all.add_subplot(131); ax2 = fig_all.add_subplot(132)
        ax3 = fig_all.add_subplot(133)
        draw_cluster(ax1); draw_corr(fig_all, ax2); draw_anom(ax3)
        fig_all.suptitle(titre, fontsize=13, fontweight="bold", y=1.01)
        fig_all.tight_layout()

        # Clustering seul
        fig_c = Figure(figsize=(9, 6))
        draw_cluster(fig_c.add_subplot(111))
        fig_c.suptitle(f"Clustering KMeans  -  {titre}", fontsize=12, fontweight="bold")
        fig_c.tight_layout()

        # Correlation seule
        fig_cr = Figure(figsize=(9, 7))
        draw_corr(fig_cr, fig_cr.add_subplot(111))
        fig_cr.suptitle(f"Matrice de Correlation  -  {titre}", fontsize=12, fontweight="bold")
        fig_cr.tight_layout()

        # Anomalies seules
        fig_a = Figure(figsize=(9, 6))
        draw_anom(fig_a.add_subplot(111))
        fig_a.suptitle(f"Anomalies  -  {titre}", fontsize=12, fontweight="bold")
        fig_a.tight_layout()

    return {"all": fig_all, "cluster": fig_c, "corr": fig_cr, "anom": fig_a}

In [ ]:
# Cellule 2 - Preprocessing (CSV)

In [ ]:
# Cellule 3 - Composants UI : Tooltip + SplashScreen

In [ ]:
# =============================================================================
#  Tooltip  -  infobulle au survol
# =============================================================================
class Tooltip:
    def __init__(self, widget, text, delay=700):
        self._w = widget; self._text = text; self._delay = delay
        self._win = None; self._job = None
        widget.bind("<Enter>", self._enter)
        widget.bind("<Leave>", self._leave)
        widget.bind("<ButtonPress>", self._leave)

    def _enter(self, e=None):
        self._job = self._w.after(self._delay, self._show)

    def _leave(self, e=None):
        if self._job: self._w.after_cancel(self._job); self._job = None
        self._hide()

    def _show(self):
        if self._win: return
        x = self._w.winfo_rootx() + 18
        y = self._w.winfo_rooty() + self._w.winfo_height() + 6
        self._win = tk.Toplevel(self._w)
        self._win.wm_overrideredirect(True)
        self._win.wm_geometry(f"+{x}+{y}")
        tk.Label(self._win, text=self._text,
                 bg="#21262d", fg="#e6edf3", font=("Segoe UI", 8),
                 relief="flat", padx=9, pady=5).pack()

    def _hide(self):
        if self._win: self._win.destroy(); self._win = None


# =============================================================================
#  SplashScreen  -  ecran d'ouverture anime
# =============================================================================
class SplashScreen:
    STEPS = [
        (12,  "Initialisation..."),
        (28,  "Chargement de NumPy - Pandas..."),
        (48,  "Initialisation de scikit-learn..."),
        (66,  "Preparation des algorithmes ML..."),
        (82,  "Configuration de l'interface..."),
        (96,  "Pret a analyser vos donnees !"),
        (100, "Lancement..."),
    ]
    C = {
        "bg":"#0d1117","panel":"#161b22","card":"#21262d","border":"#30363d",
        "fg":"#e6edf3","muted":"#8b949e","accent":"#58a6ff",
        "green":"#3fb950","yellow":"#d29922","purple":"#bc8cff","red":"#f85149",
    }

    def __init__(self, root, on_complete):
        self.root = root; self.on_complete = on_complete; self._idx = 0
        sw, sh = root.winfo_screenwidth(), root.winfo_screenheight()
        W, H = 620, 400
        self.win = tk.Toplevel(root)
        self.win.overrideredirect(True)
        self.win.geometry(f"{W}x{H}+{(sw-W)//2}+{(sh-H)//2}")
        self.win.configure(bg=self.C["bg"])
        self.win.attributes("-topmost", True); self.win.lift()
        self._build(W, H)
        self._ttk_style()
        self.root.after(180, self._tick)

    def _build(self, W, H):
        C = self.C
        tk.Frame(self.win, bg=C["accent"], height=4).pack(fill="x")
        main = tk.Frame(self.win, bg=C["bg"])
        main.pack(fill="both", expand=True)

        center = tk.Frame(main, bg=C["bg"])
        center.place(relx=0.5, rely=0.44, anchor="center")

        tk.Label(center, text="ANOMALYZE",
                 font=("Consolas", 40, "bold"),
                 bg=C["bg"], fg=C["accent"]).pack(pady=(0, 4))
        tk.Label(center,
                 text="Analyse Exploratoire  |  Detection d'Anomalies",
                 font=("Segoe UI", 11, "italic"),
                 bg=C["bg"], fg=C["muted"]).pack(pady=(0, 20))

        badges = tk.Frame(center, bg=C["bg"]); badges.pack(pady=(0, 26))
        for lbl, col in [("KMeans", C["accent"]), ("PCA", C["green"]),
                         ("Isolation Forest", C["purple"]), ("Auto-K", C["yellow"])]:
            f = tk.Frame(badges, bg=col, padx=12, pady=5)
            f.pack(side="left", padx=4)
            tk.Label(f, text=lbl, bg=col,
                     fg="#0d1117", font=("Segoe UI", 9, "bold")).pack()

        pz = tk.Frame(center, bg=C["bg"]); pz.pack(fill="x")
        self.prog = ttk.Progressbar(pz, style="Splash.Horizontal.TProgressbar",
                                     mode="determinate", maximum=100, length=440)
        self.prog.pack()
        self.slbl = tk.Label(pz, text="Initialisation...",
                             font=("Segoe UI", 9), bg=C["bg"], fg=C["muted"])
        self.slbl.pack(pady=6)

        footer = tk.Frame(main, bg=C["panel"]); footer.pack(fill="x", side="bottom")
        tk.Label(footer, text="v2.0  |  Licence 2 Informatique  |  2025-2026",
                 font=("Segoe UI", 8), bg=C["panel"], fg=C["muted"], pady=6).pack()
        tk.Frame(self.win, bg=C["green"], height=3).pack(fill="x", side="bottom")

    def _ttk_style(self):
        C = self.C; s = ttk.Style(); s.theme_use("default")
        s.configure("Splash.Horizontal.TProgressbar",
                    troughcolor=C["card"], background=C["accent"],
                    darkcolor=C["accent"], lightcolor=C["accent"],
                    bordercolor=C["panel"], thickness=10)

    def _tick(self):
        if self._idx < len(self.STEPS):
            val, msg = self.STEPS[self._idx]
            self.prog["value"] = val; self.slbl.config(text=msg)
            self._idx += 1
            delay = 320 if self._idx < len(self.STEPS) else 600
            self.root.after(delay, self._tick)
        else:
            self.root.after(200, self._close)

    def _close(self):
        self.win.destroy(); self.on_complete()

In [ ]:
# Cellule 4 - Application principale MLVisualisationApp v2

In [ ]:
# =============================================================================
#  MLVisualisationApp v2.0
#  Responsive (portable / grand ecran), onglets, export, raccourcis clavier
#  Conserve l'ossature complete de la v1
# =============================================================================
class MLVisualisationApp:

    C = {
        "bg":"#0d1117","panel":"#161b22","card":"#21262d","border":"#30363d",
        "fg":"#e6edf3","muted":"#8b949e","accent":"#58a6ff",
        "green":"#3fb950","red":"#f85149","yellow":"#d29922","purple":"#bc8cff",
    }

    def __init__(self, root):
        self.root = root
        self.root.title("ANOMALYZE v2.0  -  Apprentissage Non Supervise")
        self.root.configure(bg=self.C["bg"])

        # -- Etat (identique v1) -----------------------------------------------
        self.df             = None
        self.resultats      = None
        self.current_file   = None
        self.current_fig    = None
        self._canvas_widget = None
        self._type_courant  = None
        self._titre_courant = "ANOMALYZE"
        self._col_couleur   = None
        self._panel_vis     = True

        self._setup_window()
        self._setup_styles()
        self._build_ui()
        self._bind_keys()

    # =========================================================================
    #  Fenetre responsive
    # =========================================================================
    def _setup_window(self):
        sw = self.root.winfo_screenwidth()
        sh = self.root.winfo_screenheight()
        if   sw <= 1280: W, H = min(sw-16, 1100), min(sh-60, 660)
        elif sw <= 1440: W, H = min(sw-20, 1280), min(sh-70, 780)
        elif sw <= 1920: W, H = 1440, 900
        else:            W, H = 1600, 980
        self.root.geometry(f"{W}x{H}+{(sw-W)//2}+{max(10,(sh-H)//2)}")
        self.root.minsize(820, 560)
        self.root.resizable(True, True)

    # =========================================================================
    #  Styles ttk
    # =========================================================================
    def _setup_styles(self):
        C = self.C; s = ttk.Style(); s.theme_use("default")
        s.configure("Dark.TNotebook", background=C["bg"], borderwidth=0, tabmargins=0)
        s.configure("Dark.TNotebook.Tab",
                    background=C["card"], foreground=C["muted"],
                    padding=[16, 7], font=("Segoe UI", 9), borderwidth=0)
        s.map("Dark.TNotebook.Tab",
              background=[("selected", C["panel"]), ("active", C["border"])],
              foreground=[("selected", C["accent"]),  ("active", C["fg"])])
        s.configure("Run.Horizontal.TProgressbar",
                    troughcolor=C["card"], background=C["green"],
                    darkcolor=C["green"], lightcolor=C["green"],
                    bordercolor=C["card"], thickness=5)

    # =========================================================================
    #  Construction de l'interface
    # =========================================================================
    def _build_ui(self):
        C = self.C
        self._build_header()
        tk.Frame(self.root, bg=C["accent"], height=2).pack(fill="x")

        # PanedWindow responsive : panneau gauche | zone droite
        self._paned = tk.PanedWindow(
            self.root, orient="horizontal",
            bg=C["bg"], sashrelief="flat", sashwidth=6,
            sashpad=0, showhandle=False, relief="flat", bd=0)
        self._paned.pack(fill="both", expand=True)

        self._left_outer = tk.Frame(self._paned, bg=C["panel"])
        self._paned.add(self._left_outer, minsize=220, width=290)
        self._build_left()

        right = tk.Frame(self._paned, bg=C["bg"])
        self._paned.add(right, minsize=420)
        self._build_right(right)

        self._build_statusbar()

    # -- En-tete ---------------------------------------------------------------
    def _build_header(self):
        C = self.C
        hdr = tk.Frame(self.root, bg=C["panel"], pady=5)
        hdr.pack(fill="x")

        self._toggle_btn = tk.Button(
            hdr, text="<", command=self._toggle_panel,
            bg=C["card"], fg=C["muted"], font=("Segoe UI", 10),
            relief="flat", bd=0, cursor="hand2", padx=9, pady=2,
            activebackground=C["border"], activeforeground=C["fg"])
        self._toggle_btn.pack(side="left", padx=(8, 4))
        Tooltip(self._toggle_btn, "Masquer / Afficher le panneau  (Ctrl+B)")

        tk.Label(hdr, text="ANOMALYZE",
                 font=("Consolas", 14, "bold"),
                 bg=C["panel"], fg=C["accent"]).pack(side="left", padx=8)
        tk.Label(hdr, text="v2.0",
                 font=("Segoe UI", 8), bg=C["panel"], fg=C["muted"]).pack(side="left")
        tk.Label(hdr, text="   KMeans  |  PCA  |  Isolation Forest  |  Auto-K",
                 font=("Segoe UI", 8), bg=C["panel"], fg=C["muted"]).pack(side="left")

        for txt, cmd, tip, col in [
            ("Exporter",      self.exporter_resultats, "Ctrl+E  Enregistrer figure + CSV", C["purple"]),
            ("Reinitialiser", self.reinitialiser,       "Ctrl+R  Tout effacer",             C["yellow"]),
        ]:
            b = tk.Button(hdr, text=txt, command=cmd,
                          bg=col, fg="#0d1117" if col==C["yellow"] else "white",
                          font=("Segoe UI", 9), relief="flat", bd=0,
                          cursor="hand2", padx=12, pady=4, activebackground=col)
            b.pack(side="right", padx=4)
            Tooltip(b, tip)

    # -- Panneau gauche scrollable ---------------------------------------------
    def _build_left(self):
        C = self.C
        cvs = tk.Canvas(self._left_outer, bg=C["panel"], highlightthickness=0, bd=0)
        sb  = tk.Scrollbar(self._left_outer, orient="vertical", command=cvs.yview)
        cvs.configure(yscrollcommand=sb.set)
        cvs.pack(side="left", fill="both", expand=True)

        inner = tk.Frame(cvs, bg=C["panel"], padx=12, pady=8)
        _id   = cvs.create_window((0, 0), window=inner, anchor="nw")

        def _resize(e):
            cvs.configure(scrollregion=cvs.bbox("all"))
            if inner.winfo_reqheight() > cvs.winfo_height(): sb.pack(side="right", fill="y")
            else: sb.pack_forget()
        inner.bind("<Configure>", _resize)
        cvs.bind("<Configure>", lambda e: cvs.itemconfig(_id, width=e.width))
        cvs.bind_all("<MouseWheel>",
                     lambda e: cvs.yview_scroll(int(-1*(e.delta/120)), "units"))

        def sec(txt, ac=C["accent"]):
            f = tk.Frame(inner, bg=C["panel"]); f.pack(fill="x", pady=(12,3))
            tk.Label(f, text=txt, font=("Segoe UI", 10, "bold"),
                     bg=C["panel"], fg=C["fg"]).pack(anchor="w")
            tk.Frame(inner, bg=ac, height=1).pack(fill="x", pady=(1,4))

        def bigbtn(txt, cmd, col, tip=""):
            b = tk.Button(inner, text=txt, command=cmd,
                          bg=col, fg="white", font=("Segoe UI", 9, "bold"),
                          relief="flat", bd=0, cursor="hand2",
                          activebackground=col, pady=7)
            b.pack(fill="x", pady=2)
            if tip: Tooltip(b, tip)
            return b

        # Chargement
        sec("Charger un Dataset")
        bigbtn("Ouvrir un fichier", self.charger_fichier,
               C["accent"], "Ctrl+O  -  Fichier CSV")

        ic = tk.Frame(inner, bg=C["card"], padx=8, pady=6)
        ic.pack(fill="x", pady=(4,0))
        self.file_label = tk.Label(ic, text="Aucun fichier charge",
                                   bg=C["card"], fg=C["muted"],
                                   font=("Segoe UI", 8), wraplength=238, justify="left")
        self.file_label.pack(anchor="w")
        self.info_label = tk.Label(ic, text="", bg=C["card"], fg=C["green"],
                                   font=("Consolas", 8))
        self.info_label.pack(anchor="w")

        # Colonnes
        sec("Colonnes a analyser")
        lf = tk.Frame(inner, bg=C["card"]); lf.pack(fill="x")
        sb2 = tk.Scrollbar(lf, orient="vertical"); sb2.pack(side="right", fill="y")
        self.columns_listbox = tk.Listbox(
            lf, selectmode=tk.MULTIPLE, height=8,
            bg=C["card"], fg=C["fg"], selectbackground=C["accent"],
            font=("Consolas", 8), relief="flat", bd=0,
            activestyle="none", highlightthickness=0, yscrollcommand=sb2.set)
        sb2.config(command=self.columns_listbox.yview)
        self.columns_listbox.pack(fill="both", padx=3, pady=3)

        bf = tk.Frame(inner, bg=C["panel"]); bf.pack(fill="x", pady=(2,0))
        for txt, cmd in [("Tout", self.select_all), ("Aucun", self.deselect_all)]:
            tk.Button(bf, text=txt, command=cmd, bg=C["card"], fg=C["fg"],
                      font=("Segoe UI", 8), relief="flat", bd=0,
                      cursor="hand2", padx=8, pady=3).pack(
                      side="left", fill="x", expand=True, padx=1)

        # Parametres
        sec("Parametres")
        rk = tk.Frame(inner, bg=C["panel"]); rk.pack(fill="x", pady=3)
        tk.Label(rk, text="k  (clusters) :", bg=C["panel"], fg=C["muted"],
                 font=("Segoe UI", 9)).pack(side="left")
        self.k_var = tk.IntVar(value=3)
        sp = tk.Spinbox(rk, from_=2, to=15, textvariable=self.k_var,
                        width=4, bg=C["card"], fg=C["fg"],
                        buttonbackground=C["card"], relief="flat",
                        font=("Segoe UI", 9), highlightthickness=0)
        sp.pack(side="right")
        Tooltip(sp, "Nombre de clusters  (ignore si Auto-K coche)")

        self.auto_k_var = tk.BooleanVar(value=True)
        ck = tk.Checkbutton(inner, text="Auto-K  (silhouette score)",
                            variable=self.auto_k_var,
                            bg=C["panel"], fg=C["yellow"], selectcolor=C["card"],
                            activebackground=C["panel"], font=("Segoe UI", 9), cursor="hand2")
        ck.pack(anchor="w", pady=(2,8))
        Tooltip(ck, "Teste k=2 a 6 et choisit le meilleur Silhouette Score")

        rc = tk.Frame(inner, bg=C["panel"]); rc.pack(fill="x", pady=(0,2))
        tk.Label(rc, text="Contamination :", bg=C["panel"], fg=C["muted"],
                 font=("Segoe UI", 9)).pack(side="left")
        self.contam_var = tk.DoubleVar(value=0.05)
        self.contam_lbl = tk.Label(rc, text="5.0%", bg=C["panel"],
                                   fg=C["yellow"], font=("Consolas", 9, "bold"))
        self.contam_lbl.pack(side="right")
        sc2 = tk.Scale(inner, from_=0.01, to=0.30, resolution=0.01,
                       variable=self.contam_var, orient="horizontal",
                       bg=C["panel"], fg=C["fg"], troughcolor=C["card"],
                       highlightthickness=0, showvalue=False,
                       activebackground=C["yellow"],
                       command=self._update_contam_label)
        sc2.pack(fill="x")
        Tooltip(sc2, "Proportion estimee d'anomalies  (1% a 30%)")

        # Analyse
        sec("Analyse", C["green"])
        self._run_prog = ttk.Progressbar(inner, style="Run.Horizontal.TProgressbar",
                                          mode="indeterminate", length=240)
        self._run_prog.pack(fill="x", pady=(0,4))
        self.btn_analyse = bigbtn("Lancer l'Analyse  (F5)",
                                   self.lancer_analyse, C["green"],
                                   "F5  -  KMeans + PCA + Isolation Forest")

        # Formats
        sec("Formats acceptes")
        for icon, lbl, col in [
            ("CSV ",  "CSV  -  donnees tabulaires",   C["green"]),
        ]:
            row = tk.Frame(inner, bg=C["card"], padx=8, pady=4)
            row.pack(fill="x", pady=1)
            tk.Label(row, text=icon, bg=C["card"], fg=C["muted"],
                     font=("Consolas", 9, "bold")).pack(side="left")
            tk.Label(row, text=lbl, bg=C["card"], fg=col,
                     font=("Segoe UI", 8)).pack(side="left", padx=4)
        tk.Frame(inner, bg=C["panel"], height=14).pack()

    # -- Zone droite : onglets -------------------------------------------------
    def _build_right(self, parent):
        C = self.C
        self._nb = ttk.Notebook(parent, style="Dark.TNotebook")
        self._nb.pack(fill="both", expand=True, padx=3, pady=3)

        self._tab_all  = tk.Frame(self._nb, bg=C["bg"])
        self._tab_cl   = tk.Frame(self._nb, bg=C["bg"])
        self._tab_cor  = tk.Frame(self._nb, bg=C["bg"])
        self._tab_an   = tk.Frame(self._nb, bg=C["bg"])
        self._tab_stat = tk.Frame(self._nb, bg=C["bg"])
        self._nb.add(self._tab_all,  text="  Vue d'ensemble  ")
        self._nb.add(self._tab_cl,   text="  Clustering  ")
        self._nb.add(self._tab_cor,  text="  Correlation  ")
        self._nb.add(self._tab_an,   text="  Anomalies  ")
        self._nb.add(self._tab_stat, text="  Statistiques  ")

        for tab, msg in [
            (self._tab_all, "Vue d'ensemble"),
            (self._tab_cl,  "Clustering KMeans"),
            (self._tab_cor, "Matrice de Correlation"),
            (self._tab_an,  "Detection d'Anomalies"),
        ]:
            tk.Label(tab,
                     text=f"\n\n  {msg}  \n\nChargez un dataset\npuis lancez l'Analyse",
                     font=("Segoe UI", 11), bg=C["bg"], fg=C["muted"],
                     justify="center").pack(expand=True)

        self._build_stats_tab()

    def _build_stats_tab(self):
        C = self.C; frame = self._tab_stat
        bar = tk.Frame(frame, bg=C["panel"], pady=4); bar.pack(fill="x")
        tk.Label(bar, text="Resultats de l'analyse",
                 font=("Segoe UI", 9, "bold"),
                 bg=C["panel"], fg=C["accent"]).pack(side="left", padx=10)

        def _copy():
            self.root.clipboard_clear()
            self.root.clipboard_append(self.stats_text.get("1.0", tk.END))
            self._set_status("Stats copiees dans le presse-papiers")

        tk.Button(bar, text="Copier", command=_copy, bg=C["card"], fg=C["fg"],
                  font=("Segoe UI", 8), relief="flat", bd=0,
                  cursor="hand2", padx=8, pady=2).pack(side="right", padx=8)

        self.stats_text = tk.Text(frame, wrap=tk.WORD, font=("Consolas", 9),
                                   bg=C["card"], fg=C["fg"],
                                   insertbackground=C["fg"],
                                   relief="flat", bd=0, padx=12, pady=8,
                                   state="disabled")
        sb_s = tk.Scrollbar(frame, command=self.stats_text.yview)
        self.stats_text.configure(yscrollcommand=sb_s.set)
        sb_s.pack(side="right", fill="y")
        self.stats_text.pack(fill="both", expand=True)

        self.stats_text.tag_configure("head", foreground=C["accent"], font=("Consolas",9,"bold"))
        self.stats_text.tag_configure("sec",  foreground=C["green"],  font=("Consolas",9,"bold"))
        self.stats_text.tag_configure("val",  foreground=C["yellow"])
        self.stats_text.tag_configure("err",  foreground=C["red"])
        self.stats_text.tag_configure("dim",  foreground=C["muted"])
        self.stats_text.tag_configure("norm", foreground=C["fg"])

    # -- Barre de statut -------------------------------------------------------
    def _build_statusbar(self):
        C = self.C
        bar = tk.Frame(self.root, bg=C["panel"], pady=3); bar.pack(fill="x", side="bottom")
        self._svar = tk.StringVar(value="Pret  -  Chargez un dataset pour commencer")
        self._slbl = tk.Label(bar, textvariable=self._svar,
                              bg=C["panel"], fg=C["muted"],
                              font=("Segoe UI", 8), anchor="w")
        self._slbl.pack(side="left", padx=10)
        tk.Label(bar, text="Ctrl+O Ouvrir  |  F5 Analyser  |  Ctrl+E Exporter  |  Ctrl+R Reset  |  Ctrl+B Panneau",
                 bg=C["panel"], fg=C["border"], font=("Segoe UI", 7)).pack(side="right", padx=10)

    # =========================================================================
    #  Raccourcis clavier
    # =========================================================================
    def _bind_keys(self):
        r = self.root
        r.bind("<Control-o>", lambda e: self.charger_fichier())
        r.bind("<Control-O>", lambda e: self.charger_fichier())
        r.bind("<F5>",        lambda e: self.lancer_analyse())
        r.bind("<Control-r>", lambda e: self.reinitialiser())
        r.bind("<Control-R>", lambda e: self.reinitialiser())
        r.bind("<Control-e>", lambda e: self.exporter_resultats())
        r.bind("<Control-E>", lambda e: self.exporter_resultats())
        r.bind("<Control-b>", lambda e: self._toggle_panel())
        r.bind("<Control-B>", lambda e: self._toggle_panel())

    # =========================================================================
    #  Helpers
    # =========================================================================
    def _set_status(self, msg, col=None):
        self._svar.set(msg); self._slbl.config(fg=col or self.C["muted"])

    def _toggle_panel(self):
        if self._panel_vis:
            self._paned.paneconfig(self._left_outer, minsize=0)
            self._paned.sash_place(0, 0, 0)
            self._toggle_btn.config(text=">"); self._panel_vis = False
        else:
            self._paned.paneconfig(self._left_outer, minsize=220)
            self._paned.sash_place(0, 290, 0)
            self._toggle_btn.config(text="<"); self._panel_vis = True

    def _update_contam_label(self, val=None):
        self.contam_lbl.config(text=f"{float(self.contam_var.get())*100:.1f}%")

    def select_all(self):   self.columns_listbox.selection_set(0, tk.END)
    def deselect_all(self): self.columns_listbox.selection_clear(0, tk.END)

    def _write_stats_rich(self, segs):
        self.stats_text.configure(state="normal")
        self.stats_text.delete("1.0", tk.END)
        for item in segs:
            t, g = item if isinstance(item, tuple) else (item, "norm")
            self.stats_text.insert(tk.END, t, g)
        self.stats_text.configure(state="disabled")

    # =========================================================================
    #  Chargement des fichiers  (identique v1)
    # =========================================================================
    def charger_fichier(self):
        fp = filedialog.askopenfilename(
            title="Selectionner un dataset",
            filetypes=[("CSV","*.csv"),("Tous","*.*")])
        if not fp: return
        self._set_status(f"Chargement de {os.path.basename(fp)}...", self.C["yellow"])
        self.root.update()
        try:
            try:    df = pd.read_csv(fp)
            except: df = pd.read_csv(fp, encoding="latin1")
            self.df = df; self._type_courant = "csv"
            self._titre_courant = os.path.basename(fp); self._col_couleur = None
            self.file_label.config(text=f"CSV: {os.path.basename(fp)}", fg=self.C["green"])
            self.info_label.config(text=f"{len(df)} lignes | {df.shape[1]} col")
            num = df.select_dtypes(include=[np.number]).columns.tolist()
            self._remplir_colonnes(num); self._afficher_apercu(df, self._titre_courant)
            self.current_file = fp
            self._set_status(f"Fichier charge avec succes  :  {os.path.basename(fp)}", self.C["green"])
            self._nb.select(4)
        except Exception as e:
            messagebox.showerror("Erreur chargement", str(e))
            self._set_status(f"Erreur : {e}", self.C["red"])

    def _remplir_colonnes(self, cols):
        self.columns_listbox.delete(0, tk.END)
        for c in cols: self.columns_listbox.insert(tk.END, c)
        for i in range(min(5, len(cols))): self.columns_listbox.selection_set(i)

    def _afficher_apercu(self, df, titre):
        seg = [
            (f"{'='*52}\n", "head"),
            (f"APERCU  :  {titre}\n", "head"),
            (f"{'='*52}\n\n", "head"),
            ("Lignes      : ", "sec"), (f"{len(df)}\n", "val"),
            ("Colonnes    : ", "sec"), (f"{df.shape[1]}\n", "val"),
            ("NaN total   : ", "sec"), (f"{df.isnull().sum().sum()}\n\n", "val"),
            (f"{'-'*52}\n", "dim"),
            ("PREMIERES LIGNES\n", "sec"),
            (f"{'-'*52}\n", "dim"),
            (df.head(5).to_string() + "\n", "norm"),
        ]
        self._write_stats_rich(seg)

    # =========================================================================
    #  Analyse  (identique v1 + integration onglets + progress bar)
    # =========================================================================
    def lancer_analyse(self):
        if self.df is None:
            messagebox.showwarning("Aucun fichier", "Chargez d'abord un dataset."); return
        sel = self.columns_listbox.curselection()
        if not sel or len(sel) < 2:
            messagebox.showwarning("Colonnes", "Selectionnez au moins 2 colonnes."); return
        self.btn_analyse.configure(state="disabled", text="Analyse en cours...")
        self._run_prog.start(10)
        self._set_status("Analyse ML en cours...", self.C["yellow"])
        threading.Thread(target=self._analyse_thread, args=(sel,), daemon=True).start()

    def _analyse_thread(self, sel_idx):
        try:
            cols  = [self.columns_listbox.get(i) for i in sel_idx]
            X_df  = self.df[cols].dropna()
            if len(X_df) < 10:
                messagebox.showwarning("Donnees insuffisantes", "Moins de 10 observations."); return
            k      = self.k_var.get()
            auto   = self.auto_k_var.get()
            contam = float(self.contam_var.get())
            self.resultats  = analyser_data(X_df, k=k, contamination=contam, auto_k=auto)
            figs             = creer_visualisations_v2(self.resultats, self._titre_courant)
            self.current_fig = figs["all"]
            self.root.after(0, self._afficher_onglets, figs)
            self.root.after(0, self._afficher_stats_rich)
        except Exception as e:
            messagebox.showerror("Erreur analyse", str(e))
            import traceback; traceback.print_exc()
            self.root.after(0, lambda: self._set_status(f"Erreur : {e}", self.C["red"]))
        finally:
            self.root.after(0, self._fin_analyse)

    def _fin_analyse(self):
        self._run_prog.stop()
        self.btn_analyse.configure(state="normal", text="Lancer l'Analyse  (F5)")
        self._set_status("Analyse terminee  -  Parcourez les onglets", self.C["green"])
        self._nb.select(0)

    def _afficher_onglets(self, figs):
        for tab, fig in [(self._tab_all,figs["all"]),(self._tab_cl,figs["cluster"]),
                         (self._tab_cor,figs["corr"]),(self._tab_an,figs["anom"])]:
            for w in tab.winfo_children(): w.destroy()
            canvas = FigureCanvasTkAgg(fig, master=tab)
            canvas.draw()
            nav = NavigationToolbar2Tk(canvas, tab, pack_toolbar=False)
            nav.config(background=self.C["panel"]); nav.update()
            nav.pack(side="bottom", fill="x")
            canvas.get_tk_widget().pack(fill="both", expand=True)

    def _afficher_stats_rich(self):
        r = self.resultats
        if r is None: return
        cl = ""
        for i in range(r["k"]):
            cnt = (r["labels"]==i).sum(); pct = cnt/len(r["labels"])*100
            cl += f"  Cluster {i:2d} | {'#'*int(pct/5):<20} {cnt:5d}  ({pct:5.1f}%)\n"
        sil = r["silhouette_score"]
        s_tag  = "val" if sil>0.5 else ("sec" if sil>0.3 else "err")
        s_note = "Excellent" if sil>0.5 else ("Acceptable" if sil>0.3 else "Faible")
        v = r["variance_explained"]
        seg = [
            (f"{'='*55}\n","head"),("RESULTATS  -  ANOMALYZE v2.0\n","head"),
            (f"{'='*55}\n\n","head"),
            ("DONNEES\n","sec"),
            ("  Observations  : ","norm"),(f"{len(r['X_df'])}\n","val"),
            ("  Features      : ","norm"),(f"{len(r['X_df'].columns)}\n","val"),
            ("  Colonnes      : ","norm"),
            (f"{chr(44).join(r['X_df'].columns[:6])}{'...' if len(r['X_df'].columns)>6 else ''}\n\n","dim"),
            ("CLUSTERING  KMeans\n","sec"),
            ("  k              : ","norm"),(f"{r['k']}\n","val"),
            ("  Silhouette     : ","norm"),(f"{sil:.4f}","s_tag"),
            (f"  ({s_note})\n\n","dim"),
            (cl+"\n","norm"),
            ("PCA\n","sec"),
            ("  Variance expliquee : ","norm"),(f"{v*100:.1f}%\n\n","val" if v>0.6 else "err"),
            ("ISOLATION FOREST\n","sec"),
            ("  Anomalies detectees : ","norm"),
            (f"{r['n_anomalies']}  ({r['n_anomalies']/len(r['X_df'])*100:.2f}%)\n\n","err"),
            (f"{'-'*55}\n","dim"),("STATISTIQUES DESCRIPTIVES\n","sec"),
            (f"{'-'*55}\n","dim"),
            (r["X_df"].describe().round(3).to_string()+"\n","norm"),
        ]
        # Corriger le tag dynamique
        for i, item in enumerate(seg):
            if isinstance(item, tuple) and item[1] == "s_tag":
                seg[i] = (item[0], s_tag)
        self._write_stats_rich(seg)

    # =========================================================================
    #  Reinitialisation
    # =========================================================================
    def reinitialiser(self):
        self.df = self.resultats = self.current_file = None
        self._type_courant = None; self._titre_courant = "ANOMALYZE"
        self.columns_listbox.delete(0, tk.END)
        self.file_label.config(text="Aucun fichier charge", fg=self.C["muted"])
        self.info_label.config(text="")
        for tab, msg in [(self._tab_all,"Vue d'ensemble"),(self._tab_cl,"Clustering"),
                         (self._tab_cor,"Correlation"),(self._tab_an,"Anomalies")]:
            for w in tab.winfo_children(): w.destroy()
            tk.Label(tab, text=f"\n\n  {msg}  \n\nChargez un dataset puis lancez l'Analyse",
                     font=("Segoe UI",11), bg=self.C["bg"], fg=self.C["muted"],
                     justify="center").pack(expand=True)
        self._write_stats_rich([])
        self._set_status("Application reinitialisee  -  Chargez un dataset")

    # =========================================================================
    #  Export  PNG + PDF + CSV  (nouveau v2)
    # =========================================================================
    def exporter_resultats(self):
        if self.resultats is None:
            messagebox.showinfo("Export", "Lancez d'abord une analyse."); return
        out_dir = filedialog.askdirectory(title="Dossier d'export")
        if not out_dir: return
        base = os.path.splitext(os.path.basename(self.current_file or "anomalyze"))[0]
        try:
            if self.current_fig:
                self.current_fig.savefig(
                    os.path.join(out_dir, f"{base}_analyse.png"), dpi=150,
                    bbox_inches="tight", facecolor="white")
                self.current_fig.savefig(
                    os.path.join(out_dir, f"{base}_analyse.pdf"),
                    bbox_inches="tight", facecolor="white")
            r   = self.resultats; out = r["X_df"].copy()
            out["cluster"]  = r["labels"]
            out["anomalie"] = (r["anomalies"] == -1).astype(int)
            out.to_csv(os.path.join(out_dir, f"{base}_resultats.csv"), index=False)
            with open(os.path.join(out_dir, f"{base}_stats.txt"), "w", encoding="utf-8") as f:
                f.write(self.stats_text.get("1.0", tk.END))
            self._set_status(f"Export reussi  ->  {out_dir}", self.C["green"])
            messagebox.showinfo("Export reussi",
                f"Fichiers enregistres dans :\n{out_dir}\n\n"
                f"- {base}_analyse.png\n- {base}_analyse.pdf\n"
                f"- {base}_resultats.csv\n- {base}_stats.txt")
        except Exception as e:
            messagebox.showerror("Erreur export", str(e))
            self._set_status(f"Erreur export : {e}", self.C["red"])

In [ ]:
# Cellule 5 - Lancement avec ecran d'ouverture

In [ ]:
def lancer_app():
    root = tk.Tk()
    root.withdraw()   # Masquer pendant le splash

    def on_done():
        root.deiconify()
        root.focus_force()
        MLVisualisationApp(root)

    SplashScreen(root, on_done)
    root.mainloop()


# -- Lancement automatique en Jupyter ou script --------------------------------
if __name__ == "__main__" or True:
    lancer_app()